In [2]:
!pip install pymupdf sentence-transformers scikit-learn pandas --quiet

In [3]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

Saving sample-service-manual 1.pdf to sample-service-manual 1.pdf


In [16]:
import fitz
import re
import json
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text("text") + "\n"
    doc.close()
    return text

raw_text = extract_text(pdf_path)

In [18]:
def extract_torque_section(text):
    pattern = r"Torque Specifications(.*?)(SECTION|\Z)"
    match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)

    if match:
        return match.group(1)
    return ""

In [27]:
def extract_torque_rows(text):
    rows = []
    lines = text.split("\n")

    for line in lines:
        if "Nm" not in line:
            continue

        # Remove instruction-style lines
        junk_words = ["use", "install", "tighten", "remove",
                      "must", "should", "capacity", "minimum",
                      "maximum", "wrench"]

        if any(word in line.lower() for word in junk_words):
            continue

        # Match short component-like pattern
        match = re.search(r"^([A-Za-z][A-Za-z\s\-\(\)\/]+?)\s+(\d+)\s*Nm", line)

        if match:
            component = match.group(1).strip()
            value = match.group(2)

            # Avoid long sentences
            if len(component.split()) > 8:
                continue

            rows.append({
                "component": component,
                "spec_type": "Torque",
                "value": value,
                "unit": "Nm"
            })

    return rows

In [29]:
torque_rows = extract_torque_rows(raw_text)

In [30]:
embed_model = SentenceTransformer("intfloat/e5-small-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
components = [row["component"] for row in torque_rows]

component_embeddings = embed_model.encode(
    ["passage: " + comp for comp in components],
    convert_to_numpy=True
)

In [32]:
def search_component(query, top_k=1):
    query_embedding = embed_model.encode(
        ["query: " + query],
        convert_to_numpy=True
    )

    similarities = cosine_similarity(query_embedding, component_embeddings)[0]

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = [torque_rows[i] for i in top_indices]

    return results

In [33]:
queries = [
    "Lower ball joint nut torque",
    "Brake disc shield bolts torque",
    "Stabilizer bar bracket nuts torque"
]

for query in queries:
    print("\nQuery:", query)
    results = search_component(query)
    print(json.dumps(results, indent=2))


Query: Lower ball joint nut torque


ValueError: Expected 2D array, got 1D array instead:
array=[].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.